# Hợp nhất và tăng cường dữ liệu

Notebook đọc các CSV trong `data/` theo chunk, tạo dữ liệu phe sáng bằng phép đối xứng bàn cờ và ghi các tập dẫn xuất mà không giữ toàn bộ dữ liệu trong RAM.

## 1. Cấu hình

Pipeline nặng chỉ chạy khi đặt `RUN_DATA_PIPELINE=1`. Mặc định notebook chỉ kiểm tra nguồn và minh họa phép biến đổi.

In [1]:
from pathlib import Path
import os

import pandas as pd

DATA_DIR = Path('data')
SOURCE_FILES = sorted(DATA_DIR.glob('dark_*.csv'))
DARK_FILE = DATA_DIR / 'dark.csv'
LIGHT_FILE = DATA_DIR / 'light.csv'
COMBINED_FILE = DATA_DIR / 'animal_chess.csv'
CHUNK_SIZE = int(os.getenv('CSV_CHUNK_SIZE', '250000'))
RUN_PIPELINE = os.getenv('RUN_DATA_PIPELINE') == '1'

if not SOURCE_FILES:
    raise FileNotFoundError('Không tìm thấy data/dark_*.csv')
[(path.name, round(path.stat().st_size / 1024**3, 2)) for path in SOURCE_FILES]

[('dark_1.csv', 1.75), ('dark_2.csv', 1.75), ('dark_3.csv', 0.36)]

## 2. Hàm biến đổi

Các hàm dưới đây bảo toàn schema và đảo đồng thời vị trí, phe, ký hiệu quân, điểm số.

In [2]:
MOVE_TRANSLATION = str.maketrans('ABCDEFG123456789', 'GFEDCBA987654321')
INVERT_COLUMNS = ['side', 'trap', 'den', 'score', 'winner']

def transform_board(board: str) -> str:
    # Đảo thứ tự 63 ô và đổi hoa/thường để hoán đổi hai phe.
    if len(board) != 63:
        raise ValueError(f'Bàn cờ phải có 63 ký tự, nhận {len(board)}')
    return board[::-1].swapcase()

def transform_move(move: str) -> str:
    # A1G9 <-> G9A1 theo phép quay 180 độ.
    if len(move) != 4:
        raise ValueError(f'Nước đi không hợp lệ: {move!r}')
    return move.translate(MOVE_TRANSLATION)

def transform_chunk(dark: pd.DataFrame) -> pd.DataFrame:
    light = dark.copy()
    light['board'] = dark['board'].map(transform_board)
    light['move'] = dark['move'].map(transform_move)
    light['piece'] = dark['piece'].str.swapcase()
    light[INVERT_COLUMNS] = -dark[INVERT_COLUMNS]
    return light

In [3]:
sample = pd.read_csv(SOURCE_FILES[0], nrows=5)
sample_light = transform_chunk(sample)
pd.DataFrame({
    'board_dark': sample['board'].str[:14] + '…',
    'board_light': sample_light['board'].str[:14] + '…',
    'move_dark': sample['move'],
    'move_light': sample_light['move'],
    'score_dark': sample['score'],
    'score_light': sample_light['score'],
})

,board_dark,board_light,move_dark,move_light,score_dark,score_light
0,l-r---E-T-d---…,l-----E-T-dr--…,G7F7,A3B3,0,0
1,l--r--E-T-d---…,l-----E-T-dr--…,A3A4,G7G6,0,0
2,l--r--E-T-d---…,l-r---E-T-d---…,F7G7,B3A3,0,0
3,l---r-E-T-d---…,l-r---E-T-d---…,A4A5,G6G5,0,0
4,l---r--ET-d---…,l-r---E-T-d---…,A7A8,G3G2,0,0


## 3. Ghi dữ liệu theo chunk

File đích được xóa trước một lần; mỗi chunk sau đó được nối tiếp với header chính xác.

In [4]:
def merge_and_augment(source_files: list[Path]) -> dict[str, int]:
    outputs = (DARK_FILE, LIGHT_FILE, COMBINED_FILE)
    for output in outputs:
        output.unlink(missing_ok=True)

    counts = {'dark': 0, 'light': 0, 'combined': 0}
    for source in source_files:
        for dark in pd.read_csv(source, chunksize=CHUNK_SIZE):
            light = transform_chunk(dark)
            dark.to_csv(DARK_FILE, mode='a', header=not DARK_FILE.exists(), index=False)
            light.to_csv(LIGHT_FILE, mode='a', header=not LIGHT_FILE.exists(), index=False)

            # Tập huấn luyện không cần UUID của từng ván.
            combined = pd.concat(
                [dark.drop(columns='id'), light.drop(columns='id')],
                ignore_index=True,
            )
            combined.to_csv(
                COMBINED_FILE, mode='a', header=not COMBINED_FILE.exists(), index=False
            )
            counts['dark'] += len(dark)
            counts['light'] += len(light)
            counts['combined'] += len(combined)
    return counts

if RUN_PIPELINE:
    print(merge_and_augment(SOURCE_FILES))
else:
    print('Chế độ preview: đặt RUN_DATA_PIPELINE=1 để tạo ba file dẫn xuất.')

Chế độ preview: đặt RUN_DATA_PIPELINE=1 để tạo ba file dẫn xuất.


## Tóm tắt

- Đọc theo chunk để giới hạn RAM.
- Kiểm tra độ dài bàn cờ/nước đi trước khi biến đổi.
- Dữ liệu gốc trong `data/dark_*.csv` không bị chỉnh sửa.
- Ba file đầu ra là `dark.csv`, `light.csv` và `animal_chess.csv`.